In [ ]:
import pandas as pd 
import numpy as np

### Data Preperation

In [ ]:
Mangrove_Extent = pd.read_csv("../data/Mangrove_Extent.csv")
Mangrove_Extent_Upper95 = pd.read_csv("../data/Mangrove_Extent_Upper95.csv")
Mangrove_Extent_Lower95 = pd.read_csv("../data/Mangrove_Extent_Lower95.csv")

In [ ]:
Mangrove_Extent.head(5)

In [ ]:
Mangrove_Extent.info()

In [ ]:
Mangrove_Extent.isna().sum()

In [ ]:
Mangrove_Extent.describe() 

In [ ]:
Mangrove_Extent.dtypes

In [ ]:
### Removes the commas and converts the string vaues to numerical values
Mangrove_Extent_num = Mangrove_Extent.iloc[:,2:43].apply(
    lambda key: pd.to_numeric(key.str.replace(',',''))
) 
Mangrove_Extent_num.dtypes

In [ ]:
# Calculate annual differences across all year columns
mangrove_difference = Mangrove_Extent_num.diff(axis=1)

In [ ]:
Mangrove_Extent.columns

In [ ]:
# testing the difference calculations 

# testing: 1986 difference
assert (
    Mangrove_Extent_num.iloc[0,1] - Mangrove_Extent_num.iloc[0,0]
    == mangrove_difference.iloc[0,1]
)

#testing: 1987 difference 
assert (
    Mangrove_Extent_num.iloc[0,2] - Mangrove_Extent_num.iloc[0,1]
    == mangrove_difference.iloc[0,2]
)

#testing: 1987 difference 
assert (
    Mangrove_Extent_num.iloc[0, 3] - Mangrove_Extent_num.iloc[0, 2]
    == mangrove_difference.iloc[0, 3]
)

#testing that 1985 difference is NaN 
assert mangrove_difference.iloc[:,0].isna().all()

### Annual changes across countries 

1. Max absolute annual increase
2. max absolute annual decrease
3. max percentage annual increase
4. max percentage annual decrease
5. standard deviation of annual change



### Largest annual increase for each country 

In [ ]:
# Maximum for each country 
largest_increase = mangrove_difference.max(axis=1)
largest_increase.isna().sum() # 2 Na values 
# country with largest increase globally 
largest_increase.idxmax() # returns 55
Mangrove_Extent.iloc[55] # indonesia 

### Largest Annual decrease for each country 

In [ ]:
# minimum for each country 
largest_decrease = mangrove_difference.min(axis=1) 
largest_decrease.idxmin()
Mangrove_Extent.iloc[55]

In [ ]:
# largest annual losses 
largest_decrease_year = mangrove_difference.min().idxmin()
mangrove_difference[largest_decrease_year].idxmin()


### Annual Variability 

In [ ]:
annual_variability = mangrove_difference.std(axis=1)
annual_variability.idxmax()


In [ ]:
print("Largest annual increases:")
print(largest_increase.describe())

print("Largest annual decreases:")
print(largest_decrease.describe())

print("Annual variability:")
print(annual_variability.describe())

In [ ]:
Mangrove_Extent.iloc[[129, 130]][['iso', 'cnty_name']]

# Percentage changes 

### Annual percentage change for each country 

In [ ]:
mangrove_pct_change = Mangrove_Extent_num.pct_change(axis=1, fill_method=None) * 100
mangrove_pct_change = mangrove_pct_change.replace([np.inf, -np.inf], np.nan) # so we dont need to calcualte individually greatest increase and decrease we just find the 
## max and minimum from the pct chang evalues because pct change calculates values both ways increases and decreases? 

In [ ]:
mangrove_pct_change.head(10)

In [ ]:
# max annual percentage per country 
largest_pct_increase = mangrove_pct_change.max(axis=1)

# max percent decrease per country 
largest_pct_decrease = mangrove_pct_change.min(axis=1)

In [ ]:
largest_pct_increase.describe()

In [ ]:
largest_pct_decrease.describe()

In [ ]:
largest_pct_increase.idxmax()

In [ ]:
largest_pct_decrease.idxmin()

In [ ]:
## Largest Percent Increse country Cocos (Keeling Islands)
Mangrove_Extent.iloc[20]

In [ ]:
# largest percent decreade country 
Mangrove_Extent.iloc[85]

In [ ]:
## Cocos Island had a 400 percent increase but the absolute increase was only 4 units in the dataset

## nauru had a 100 percent decrease but the absolute decrease was only 1 unit. 

# countries with much larger mangrove areas could expirience substantial absolute loss while have a smaller percentage decrease
# percent changes can exaggerate the important of changes in countries with very small mangrove extents to begin with. 

In [ ]:
# Exclude the overall total row
country_rows = Mangrove_Extent["cnty_name"] != "Total (Ha)"

# Get the 10 largest increases among countries only
top_10_increases = largest_increase[country_rows].nlargest(10)

# Replace row indexes with country names
top_10_increases.index = Mangrove_Extent.loc[top_10_increases.index, "cnty_name"]

# Plot
ax = top_10_increases.plot(kind="bar")

ax.set_xlabel("Country")
ax.set_ylabel("Mangrove Area Increase (Hecaters)")
ax.set_title("Top 10 Countries by Largest Annual Mangrove Increase")
ax.tick_params(axis="x", labelrotation=45)

In [ ]:
top_10_increases